In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Magnitude-aware community weighting — Italy INGV Catalog (1985-2025)

Communities are usually ranked by node count alone. Here we re-weight them by
folding in the magnitude of the events they contain, so the analysis
concentrates on the big, energetic communities instead of the many tiny ones.
Three weighting schemes are compared, on three base networks (soft → hard →
standard Abe-Suzuki).
"""

import logging
from pathlib import Path

import networkx as nx
import pandas as pd
import plotly.io as pio
import seaborn as sns
from shapely.geometry import Point, Polygon

from src.network import discretize_space_3d
from src.network_custom import build_abe_suzuki_network_custom_hybrid
from src.community_custom import (
    run_louvain_directed_hybrid,
    run_infomap_hybrid,
    run_hdbscan_geo_hybrid,
)
from src.community_weight import (
    aggregate_community_stats,
    weight_gr_energy,
    weight_count_mag,
    weight_size_exp,
    rank_top_k,
    plot_weight_bars,
    plot_weighted_community_geo,
)
from src.plotutils import setup_matplotlib, configure_saves

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
# Silence chatty third-party loggers (Kaleido/Chromium PDF export spam, matplotlib
# font cache, PIL image debug, asyncio cleanup) — keep our own INFO messages visible.
for _noisy in ("kaleido", "choreographer", "logistro", "matplotlib",
               "PIL", "urllib3", "asyncio"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

# ── Global configuration ─────────────────────────────────────────────────────
DATA_DIR    = Path("data/INGV")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data").mkdir(exist_ok=True)
CUT_YEAR   = 1985
TARGET_CRS = "epsg:32632"             # UTM Zone 32N — Italy

TOP_K      = 10                       # keep this many highest-weight communities
ALPHA      = 1.0                      # magnitude exponent for the size×10^(αM) scheme
MIN_CELLS  = 2                        # drop communities smaller than this before ranking
MIN_EVENTS = 10                       # …and with fewer events (excludes singleton/tiny ones)

# Hybrid network parameters — Optuna TPE-optimal values
# (see network_hybrid_optuna.py). NMI(Louvain↔InfoMap) = 0.758 at these params.
CELL_SIZE          = 45
HYBRID_ALPHA       = 0.5391
HYBRID_TAU_DAYS    = 0.8561
HYBRID_R0          = 20.331
HYBRID_SPATIAL_KM  = 300.0     # fixed during Optuna search
HYBRID_TIME_HOURS  = 24        # fixed during Optuna search

# Italy map presentation
_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

setup_matplotlib()
configure_saves(True, True, RESULTS_DIR / "figures" / "italy" / "community_weight")

# The three weighting schemes: (label, function, short weight symbol used on axes).
# The third scheme (n_cells · 10^(α·M̄)) works well on InfoMap and HDBSCAN-geo
# (clean macro-partitions, no long fragment tail) but is sensitive to small
# high-magnitude fragments produced by Louvain — interpret the Louvain panel
# of this scheme as a methodological-sensitivity figure rather than a primary
# result.
WEIGHT_SCHEMES = [
    ("GR energy",       weight_gr_energy,                       "Σ 10^(1.5·M)"),
    ("n_events × M̄",    weight_count_mag,                       "n_events · M̄"),
    ("size × 10^(α·M̄)", lambda s: weight_size_exp(s, ALPHA),    "n_cells · 10^(α·M̄)"),
]

# Community-detection methods applied to every network: a flow-based method
# (InfoMap — conceptually correct for a directed transition network) and a
# structural one (directed Louvain via Leiden). Each is weighted independently.
DETECTION_METHODS = [
    ("Louvain",     lambda G: run_louvain_directed_hybrid(G, resolution=1.0)),
    ("InfoMap",     lambda G: run_infomap_hybrid(G, directed=True, seed=42)),
    ("HDBSCAN-geo", lambda G: run_hdbscan_geo_hybrid(
        G, min_cluster_size=10, target_crs=TARGET_CRS)),
]


def weighted_community_report(net_name, build_fn, cell_size, df_net,
                              top_k=TOP_K, methods=None):
    """
    Build a network, detect communities with each method in ``methods``
    (default: directed Louvain + InfoMap), then rank the communities of each
    partition under all three magnitude-aware weighting schemes.

    Returns ``(G_giant, results)`` where ``results`` maps method name →
    ``(community_map, stats)``. Writes one CSV per (network, method) plus, per
    scheme, a top-K table, a weighted bar chart and a weighted geo map.
    """
    methods = methods or DETECTION_METHODS
    print(f"\n{'='*70}\n{net_name}\n{'='*70}")
    G = build_fn(df_net)
    wcc = list(nx.weakly_connected_components(G))
    G_giant = G.subgraph(max(wcc, key=len)).copy()
    print(f"{G.number_of_nodes():,} nodes / {G.number_of_edges():,} edges | "
          f"giant {G_giant.number_of_nodes():,} "
          f"({100*G_giant.number_of_nodes()/G.number_of_nodes():.1f}%)")

    # Magnitudes aggregated from the SAME df at the SAME cell size → ids align.
    df_grid = discretize_space_3d(df_net, cell_size, target_crs=TARGET_CRS, info=False)
    # Per-cell event counts — used by plot_weighted_community_geo to size each
    # cell's marker by intrinsic activity (decouples colour=community from
    # size=activity, avoids the "uniform-tile per community" rendering).
    cell_event_counts = df_grid.groupby("cell_id").size().to_dict()

    results = {}
    for method_name, detect_fn in methods:
        community_map = detect_fn(G_giant)
        print(f"\n{method_name}: {len(set(community_map.values()))} communities")
        stats = aggregate_community_stats(df_grid, community_map)

        # All three weights side-by-side in one table for the CSV.
        stats_all = stats.copy()
        for label, fn, _ in WEIGHT_SCHEMES:
            stats_all[f"w_{_slugify(label)}"] = fn(stats)["weight"].values
        out_csv = (RESULTS_DIR / "data"
                   / f"community_weights_{_slugify(net_name)}_{_slugify(method_name)}.csv")
        stats_all.sort_values("energy_sum", ascending=False).to_csv(out_csv, index=False)
        print(f"Wrote {out_csv}")

        for label, fn, symbol in WEIGHT_SCHEMES:
            ranked = rank_top_k(fn(stats), k=top_k,
                                min_cells=MIN_CELLS, min_events=MIN_EVENTS)
            print(f"\n--- {net_name} | {method_name} | weight = {label} ({symbol}) "
                  f"— top {top_k} ---")
            display(ranked[["rank", "community", "n_cells", "n_events",
                            "mean_mag", "max_mag", "weight"]])
            tag = f"{_slugify(net_name)}_{_slugify(method_name)}_{_slugify(label)}"
            plot_weight_bars(
                ranked, title=f"{net_name} — {method_name} — {label}",
                weight_label=symbol, save_name=f"community_weight_bars_{tag}",
            )
            plot_weighted_community_geo(
                G_giant, community_map, ranked,
                title=net_name, weight_label=symbol,
                method_name=f"{method_name} — {label}",
                bounds=_IT_BOUNDS, height=MAP_HEIGHT, width=MAP_WIDTH,
                cell_event_counts=cell_event_counts,
                save_name=f"community_weighted_geo_{tag}",
            )

        results[method_name] = (community_map, stats)

    return G_giant, results


def _slugify(s):
    return (s.lower()
            .replace(" ", "_").replace("×", "x").replace("·", "")
            .replace("^", "").replace("(", "").replace(")", "")
            .replace("ā", "a").replace("α", "alpha").replace("σ", "sum")
            .replace(".", "").replace("__", "_"))

## The Idea

Community detection labels every spatial cell with a community, but ranking
communities by **node count alone** treats a sprawling swarm of micro-tremors
the same as a compact cluster that hosted a mainshock. We instead weight each
community by the **magnitude of the events it contains**, so the focus lands
on the big, energetic communities and the long tail of tiny ones is dropped.

Every network is partitioned **twice** — with directed Louvain (structural)
and InfoMap (flow-based, conceptually correct for a directed transition
network) — and each partition is weighted independently, so we can see
whether the top communities agree across detection methods.

Three weighting schemes are compared (each implemented as a separate function
in ``src/community_weight.py``):
$$ w_c^{\text{energy}} = \sum_{e \in c} 10^{1.5 M_e}, \qquad
w_c^{\text{count}}  = n_e^{(c)} \cdot \overline{M}_c, \qquad
w_c^{\text{size}}   = n_{\text{cells}}^{(c)} \cdot 10^{\alpha \overline{M}_c}. $$
The first is the Gutenberg-Richter radiated energy (*Gutenberg & Richter
1944*) — a single strong event dominates it, so mainshock communities rise to
the top. The second is a readable busy-and-strong blend. The third mirrors
the soft-network edge weighting, scaling spatial extent by an exponential of
the mean magnitude ($\alpha$ tunes how hard magnitude dominates over size).
Communities are then **ranked and the top-$K$ kept**, after dropping any with
fewer than ``MIN_CELLS`` cells or ``MIN_EVENTS`` events — without that floor
the $\text{size} \times 10^{\alpha \overline{M}}$ scheme would let a singleton
community holding one strong shock outrank genuinely large clusters, defeating
the whole "focus on the big ones" goal.

Each scheme is applied to the **hybrid Abe-Suzuki network at Optuna's
TPE-best parameters** — the project's canonical construction (soft
magnitude/decay weights plus a hard $\Delta r$/$\Delta t$ filter to remove
spurious far/late links). The Optuna search maximised
NMI(Louvain $\leftrightarrow$ InfoMap), so the weighted-community analysis
runs on the network that produces the cleanest community structure
recoverable from this catalog.

## Data Loading

Load the INGV Italian catalog, restrict to $\text{year} \geq$ ``CUT_YEAR``
and crop to the Italian region with a Shapely polygon. The full catalog (with
magnitudes) is reused for every network so the per-community magnitude
aggregation aligns with the node cell ids.

In [ ]:
print("Loading Italy earthquake catalog...")
df = pd.read_csv(DATA_DIR / "italy_earthquakes_1985_2025.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df_net = (
    df[df["time"].dt.year >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)

_ITALY_POLYGON = Polygon([
    (13.5, 44.5), (19.0, 41.2), (19.0, 35.6), (11.7, 35.5), (11.6, 37.9),
    (7.2, 38.0), (7.2, 42.6), (5.2, 45.5), (11.6, 48.4), (16.0, 46.7), (13.5, 44.5),
])
df_net = df_net[df_net.apply(
    lambda r: _ITALY_POLYGON.contains(Point(r["longitude"], r["latitude"])), axis=1
)].reset_index(drop=True)
print(f"After polygon filter: {len(df_net):,} earthquakes  "
      f"(M {df_net['magnitude'].min():.1f}–{df_net['magnitude'].max():.1f})")

## Hybrid Network — Weighted Communities

Build the **hybrid** Abe-Suzuki network at Optuna's TPE-optimal parameters
(cell 45 km, $\alpha = 0.5391$, $r_0 = 20.331$ km, $\tau = 0.8561$ d,
with hard filters $\Delta r \leq 300$ km and $\Delta t \leq 24$ h on top
of the soft weights), detect communities with directed Louvain, InfoMap
and the HDBSCAN-geo spatial null, then rank each partition under all
three weighting schemes and keep the top-$K$. These parameters produce
the highest method-stability NMI = 0.758 on the hybrid 30 km giant —
the cleanest community structure recoverable.

In [ ]:
G_hybrid, results_hybrid = weighted_community_report(
    f"Hybrid network ({CELL_SIZE} km, Optuna best)",
    lambda d: build_abe_suzuki_network_custom_hybrid(
        d, cell_size_km=CELL_SIZE,
        spatial_threshold_km=HYBRID_SPATIAL_KM,
        time_threshold_sec=HYBRID_TIME_HOURS * 3600,
        target_crs=TARGET_CRS,
        alpha=HYBRID_ALPHA, tau=HYBRID_TAU_DAYS * 86400.0, r0=HYBRID_R0,
        info=True),
    cell_size=CELL_SIZE,
    df_net=df_net,
)